# 04. Graph RAG with Real Text

[NB3](./03_graph_rag_with_langchain.ipynb) ended with a callout. Look at what `manual_graph_rag` returned: rows of names, titles, numbers. The "answer" was the LLM rephrasing structured data into a sentence. That's a perfectly valid kind of retrieval. It is **not** what most people mean by RAG.

Production graph RAG attaches **real text** to nodes (plot summaries, script excerpts, biographies, support tickets, whatever your domain has) and the graph tells you *which* texts to pull. The graph is the **index** over the text, not a replacement for it.

This notebook makes that real. We've already downloaded eleven movie scripts from [IMSDb](https://imsdb.com) and attached each one to its `Movie` node via a `HAS_SCRIPT` relationship. Now we'll ask questions whose answers live inside those scripts.

## What changed in the graph

Same nodes and edges as before, plus two new ones:

```
                       (Person) -[:DIRECTED]-> (Movie)
                       (Person) -[:ACTED_IN]-> (Movie)
                       (Movie) -[:PRODUCED_BY]-> (Studio)
                       (Movie) -[:HAS_SCRIPT]-> (Script)   <-- whole document
                       (Movie) -[:HAS_SCENE]-> (Scene)     <-- one per slug line
```

`Script` holds the **whole** screenplay as a single string. `Scene` holds **one chunk** of that screenplay (the text between two `INT.`/`EXT.` slug lines), with its heading and scene number. Both come from the same source file, but they represent two different retrieval patterns:

- `Movie -> Script -> content` is "give me the whole document" (Act I and Act II below).
- `Movie -> Scene -> body` is "give me the relevant chunk" (Act III).

We're going to see why the first pattern feels right and then breaks, and why the second pattern is what production graph RAG actually uses. Both were built by `scripts/build_corpus.py` after `scripts/download_scripts.py` cached the eleven scripts to `data/scripts/<slug>.txt`.

## Setup

> **Run this notebook top to bottom.** Same rule as NB3: each cell builds on variables defined above it.

This notebook costs slightly more than NB3 to run because scripts are big. We'll show you how to keep that in check.

In [1]:
from helpers import load_env, get_openai_client, get_kuzu_conn
from scripts.build_corpus import build_if_missing

build_if_missing()

cfg = load_env()
client = get_openai_client(cfg)
conn = get_kuzu_conn()

# Quick sanity check: are scripts AND scenes loaded?
n_scripts = conn.execute(
    "MATCH (s:Script) RETURN count(s) AS n"
).get_as_df().iloc[0]["n"]
n_scenes = conn.execute(
    "MATCH (sc:Scene) RETURN count(sc) AS n"
).get_as_df().iloc[0]["n"]
n_expected_scripts = 11

if n_scripts < n_expected_scripts or n_scenes == 0:
    print(
        f"Setup incomplete: {n_scripts} / {n_expected_scripts} scripts, "
        f"{n_scenes} scenes. Run:\n"
        "  python scripts/download_scripts.py\n"
        "  python scripts/build_corpus.py --force"
    )
else:
    print(f"Ready. {n_scripts} scripts and {n_scenes} scenes attached to Movie nodes.")

Ready. 11 scripts and 1809 scenes attached to Movie nodes.


## Act I: read a script off a node

The simplest possible graph-with-text query. Find one movie, follow its `HAS_SCRIPT` edge, return the text. No LLM yet, just Cypher.

In [2]:
rows = conn.execute("""
    MATCH (m:Movie {title: 'Pulp Fiction'})-[:HAS_SCRIPT]->(s:Script)
    RETURN m.title AS movie, s.source_url AS source, s.content AS content
""").get_as_df()

# Don't print 297KB of script. Just the shape, the source, and a small slice.
row = rows.iloc[0]
print(f"Movie:   {row['movie']}")
print(f"Source:  {row['source']}")
print(f"Length:  {len(row['content']):,} characters")
print()
print("--- first 600 characters of the script ---")
print(row["content"][:600])

Movie:   Pulp Fiction
Source:  https://imsdb.com/scripts/Pulp-Fiction.html
Length:  296,961 characters

--- first 600 characters of the script ---
"PULP FICTION" -- by Quentin Tarantino & Roger Avary


                                      "PULP FICTION"

                                            By

                             Quentin Tarantino & Roger Avary

                

               PULP [pulp] n.

               1. A soft, moist, shapeless mass or matter.

               2. A magazine or book containing lurid subject matter and 
               being characteristically printed on rough, unfinished paper.

               American Heritage Dictionary: New College Edition

               INT. COFFEE SHOP � MORNING

            


That's a real screenplay sitting in a database column. We pulled it through a graph traversal, not a full-text search. The graph told us *which* document to read.

Now imagine the question wasn't "give me Pulp Fiction's script", it was something the model needs to *understand from* the script. That's where the LLM comes back.

## Act II: ask a question whose answer lives in the text

We reuse the four-step pattern from NB3 (generate Cypher, run it, send the result to the LLM, get an answer). The only thing that changes is what the Cypher returns. Instead of names and titles, it returns the script's `content` field.

The LLM does exactly what it would do in vector RAG: read the passage, answer the question, ground the answer in the text.

**Spoiler for Act III**: this is going to work for one kind of question and fail for another. Watch what happens.

In [3]:
from langchain_kuzu.graphs.kuzu_graph import KuzuGraph

graph = KuzuGraph(conn.database, allow_dangerous_requests=True)
schema = graph.schema
print(schema)

ALWAYS RESPECT THE RELATIONSHIP DIRECTIONS:
---
(:Person) -[:DIRECTED]-> (:Movie)
(:Person) -[:ACTED_IN]-> (:Movie)
(:Movie) -[:PRODUCED_BY]-> (:Studio)
(:Movie) -[:HAS_SCRIPT]-> (:Script)
(:Movie) -[:HAS_SCENE]-> (:Scene)
---

Node properties:
  - Person
    - name: string
    - born_year: int64
  - Movie
    - title: string
    - year: int64
  - Studio
    - name: string
  - Script
    - movie_title: string
    - source_url: string
    - content: string
  - Scene
    - scene_id: string
    - movie_title: string
    - scene_number: int64
    - heading: string
    - body: string

Relationship properties:
- DIRECTED
    - Movie: name
- ACTED_IN
    - Movie: name
- PRODUCED_BY
    - Studio: title
- HAS_SCRIPT
    - Script: title
- HAS_SCENE
    - Scene: title


Notice the `Script` node and `HAS_SCRIPT` relationship are right there in the schema. The LLM has everything it needs to write a query that pulls script text.

In [4]:
CYPHER_INSTRUCTIONS = """You are an expert in translating natural language questions into Cypher statements.
You will be provided with a question and a graph schema.
Use only the provided relationship types and properties in the schema to generate a Cypher statement.
If the question is about the content, plot, or text of a movie, your Cypher MUST traverse
from Movie to Script via the HAS_SCRIPT relationship and return s.content.
Do not include any explanations or apologies in your responses.
Output ONLY the Cypher statement."""


def generate_cypher(question: str) -> str:
    prompt = (
        CYPHER_INSTRUCTIONS
        + "\n\nSchema:\n" + schema
        + "\n\nThe question is:\n" + question
    )
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    cypher = resp.choices[0].message.content.strip()
    if cypher.startswith("```"):
        cypher = cypher.split("```")[1]
        if cypher.startswith("cypher"):
            cypher = cypher[len("cypher"):]
        cypher = cypher.strip()
    return cypher


# Question whose answer is in the OPENING of the script (within the first
# ~3K characters). The truncation in synthesize_answer_from_text won't bite us.
question = "In the Pulp Fiction script, who are the two robbers in the opening scene at the coffee shop?"
cypher = generate_cypher(question)
print(cypher)

MATCH (m:Movie {title: 'Pulp Fiction'})-[:HAS_SCRIPT]->(s:Script) RETURN s.content


### Run the Cypher

Here's a wrinkle that doesn't show up in NB3: this query returns hundreds of thousands of characters in a single cell. Pandas helpfully truncates it for display, but if we feed the raw dataframe straight into a prompt, we're paying for a *lot* of tokens. Let me show what the row looks like and how big it is.

In [5]:
rows = conn.execute(cypher).get_as_df()
print(f"Rows returned: {len(rows)}")
for col in rows.columns:
    sample = rows.iloc[0][col]
    if isinstance(sample, str):
        print(f"  {col}: {len(sample):,} chars")
    else:
        print(f"  {col}: {sample}")

Rows returned: 1
  s.content: 296,961 chars


### Synthesize the answer

The `synthesize_answer` from NB3 took rows and asked the LLM to write a sentence. It worked because the rows were small. Now they're not.

Two options:
1. **Truncate the script** to a reasonable window. Cheap, lossy.
2. **Don't pass the whole document at all.** Instead, retrieve only the relevant chunk. That's Act III.

For now we go with option 1, with eyes open. We'll truncate to ~40K characters (well within `gpt-4o-mini`'s context) and the LLM will see only the start of the script. For the question we're asking (about the opening scene), that's fine. For a question about the middle or end of the script, it won't be.

In [7]:
MAX_CONTEXT_CHARS = 40_000  # ~10K tokens, well within budget


def truncate_for_context(text: str, max_chars: int = MAX_CONTEXT_CHARS) -> tuple[str, bool]:
    if len(text) <= max_chars:
        return text, False
    return text[:max_chars] + "\n\n[... truncated ...]", True


ANSWER_INSTRUCTIONS = """You are answering a question using only the text provided.
The text may be a movie script or part of one. Do not make anything up.
If the text doesn't answer the question, say so plainly."""


def synthesize_answer_from_text(question: str, rows) -> str:
    # Pull the longest string column out as the "context" passage.
    # For our schema that's almost always s.content.
    context_parts = []
    for _, row in rows.iterrows():
        for val in row:
            if isinstance(val, str) and len(val) > 200:
                truncated, did_truncate = truncate_for_context(val)
                context_parts.append(truncated)
    context = "\n\n---\n\n".join(context_parts) if context_parts else rows.to_string(index=False)

    prompt = (
        ANSWER_INSTRUCTIONS
        + "\n\nQuestion: " + question
        + "\n\nText:\n" + context
        + "\n\nAnswer:"
    )
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


print(synthesize_answer_from_text(question, rows))

The two robbers in the opening scene at the coffee shop are referred to as Pumpkin and Honey Bunny.


That answer is qualitatively different from NB3's answers. It isn't summarizing a table of facts; it's reading a passage and explaining what it says. **That's the RAG most people mean.**

Same four-step loop. Same graph. Same LLM. The only thing that changed is what the Cypher returns: real prose instead of structured rows.

### Wrap it up

Let's bundle it into one function, same shape as NB3's `manual_graph_rag`.

In [ ]:
def manual_text_rag(question: str) -> dict:
    cypher = generate_cypher(question)
    rows = conn.execute(cypher).get_as_df()
    answer = synthesize_answer_from_text(question, rows)
    return {"question": question, "cypher": cypher, "rows_shape": rows.shape, "answer": answer}


# Another opening-scene question. Pumpkin and Honey Bunny are introduced
# in the first ~4% of the Pulp Fiction script, well within the truncation window.
result = manual_text_rag(
    "In the opening scene of Pulp Fiction, how does the script describe Pumpkin's "
    "robbery persona versus Honey Bunny's?"
)
print("Cypher generated:")
print(result["cypher"])
print()
print(f"Rows: {result['rows_shape']}")
print()
print("Answer:")
print(result["answer"])

### Now watch it break

Same function, different question. The Ezekiel speech and the sunken-place reveals happen well past the first 40K characters of their scripts. Watch the LLM tell us honestly that it can't answer.

In [9]:
failed = manual_text_rag(
    "In the Get Out script, what is the 'sunken place'?"
)
print("Cypher generated:")
print(failed["cypher"])
print()
print(f"Rows: {failed['rows_shape']}  (the row holds the full {232117:,}-char script)")
print()
print("Answer:")
print(failed["answer"])

Cypher generated:
MATCH (m:Movie)-[:HAS_SCRIPT]->(s:Script) WHERE m.title = 'Get Out' RETURN s.content

Rows: (1, 1)  (the row holds the full 232,117-char script)

Answer:
The text does not provide an explanation of what the 'sunken place' is in the Get Out script.


The model isn't doing a bad job. It's doing the *right* job with bad input. The Get Out script is 232K characters. Our truncation sends the first 40K (~17%). The Sunken Place scene is roughly halfway through the screenplay, **well outside the window we sent**. The LLM is correctly reporting that the text it received doesn't answer the question, because it doesn't.

There are two ways out:

1. **Send the whole script every time.** Works for our 11 movies (the biggest is 320K chars and `gpt-4o-mini` has 128K tokens of context). Costs ~$0.01 per question instead of ~$0.001. Scales badly: a graph with thousands of long documents won't fit.

2. **Don't retrieve the whole script. Retrieve only the relevant chunk.** The graph already has the structure for this: every `Movie` has many `Scene` nodes, each one a few thousand characters of focused text. We just have to teach the LLM to use them.

That's Act III.

## Act III: scene-level retrieval

Screenplays have a natural chunking unit built in: the scene. Every scene starts with a slug line like `INT. HOSPITAL ROOM - DAY` or `EXT - COUNTRYSIDE - NIGHT`. `scripts/build_corpus.py` already split each script on those slug lines and loaded the chunks into the graph as `Scene` nodes, attached to their Movie via `HAS_SCENE`.

That gives us a much smaller unit to retrieve. The Get Out script is 232K chars in one Script node, but **118 Scene nodes** averaging ~2K chars each. The Sunken Place reveals live in 3 specific scenes. If we can ask the LLM to find them, the answer becomes easy.

The cheap trick: let the LLM write Cypher that filters Scene bodies by keyword, returning only the matching scenes. This isn't "vector retrieval" (that's module 05). It's the simplest possible scene-level retrieval, and it works."""

In [ ]:
# The hand-written version of what we want the LLM to learn to write.
# Filter by Scene.body CONTAINS the keyword instead of returning the whole script.
# Note the lower() wrap on both sides: Kuzu's CONTAINS is case-sensitive, and
# we don't want to miss matches because the script capitalizes 'Sunken Place'
# but a user's question might not.
scene_rows = conn.execute("""
    MATCH (m:Movie {title: 'Get Out'})-[:HAS_SCENE]->(sc:Scene)
    WHERE lower(sc.body) CONTAINS lower('sunken place')
    RETURN sc.scene_number AS n, sc.heading AS heading, sc.body AS body
    ORDER BY sc.scene_number
""").get_as_df()

total_chars = sum(len(b) for b in scene_rows["body"])
print(f"Scenes retrieved: {len(scene_rows)} (out of 118 in Get Out)")
for _, row in scene_rows.iterrows():
    print(f"  scene {row['n']}: {row['heading'][:60]} ({len(row['body']):,} chars)")
print(f"Total context: {total_chars:,} chars (~{total_chars // 4:,} tokens)")
print()
print("Massively smaller than the 232K chars of the whole Get Out script.")

### Teach the LLM to use scenes

The hand-written query above used `CONTAINS` against a keyword we knew. The LLM doesn't have that hint, so we have to tell it: *prefer scene-level retrieval, and use the natural keywords from the question to filter `Scene.body`.*

A small Cypher-generation prompt change does it.

In [ ]:
SCENE_CYPHER_INSTRUCTIONS = """You are an expert in translating natural language questions into Cypher statements.
You will be provided with a question and a graph schema.

Rules:
- Use ONLY the relationship types and properties in the schema.
- For questions about the content, plot, or text of a movie, traverse
  from Movie to Scene via HAS_SCENE (NOT to Script). Filter Scene.body
  with CONTAINS using the most distinctive keyword from the question.
  Return sc.heading and sc.body.
- Kuzu's CONTAINS is case-sensitive. ALWAYS wrap both sides with lower()
  so 'Sunken Place' matches 'sunken place'. Example:
    WHERE lower(sc.body) CONTAINS lower('sunken place')
- Prefer ONE short, distinctive keyword (one or two words) over a long
  phrase. Movie scripts have unpredictable line breaks and indentation,
  so long phrases often won't match as a substring. For 'what does Jules
  say about the path of the righteous man', use 'Ezekiel' or 'righteous',
  not the full phrase.
- For structural questions (who directed, who acted in, what year),
  use the Person/Movie/Studio nodes as before.
- Do not include any explanations. Output ONLY the Cypher statement."""


def generate_scene_cypher(question: str) -> str:
    prompt = (
        SCENE_CYPHER_INSTRUCTIONS
        + "\n\nSchema:\n" + schema
        + "\n\nThe question is:\n" + question
    )
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    cypher = resp.choices[0].message.content.strip()
    if cypher.startswith("```"):
        cypher = cypher.split("```")[1]
        if cypher.startswith("cypher"):
            cypher = cypher[len("cypher"):]
        cypher = cypher.strip()
    return cypher


def manual_scene_rag(question: str) -> dict:
    cypher = generate_scene_cypher(question)
    rows = conn.execute(cypher).get_as_df()
    answer = synthesize_answer_from_text(question, rows)
    return {"question": question, "cypher": cypher, "rows_shape": rows.shape, "answer": answer}


# The previously-failed question, retried with scene-level retrieval
result = manual_scene_rag("In the Get Out script, what is the 'sunken place'?")
print("Cypher generated:")
print(result["cypher"])
print()
print(f"Rows: {result['rows_shape']}")
print()
print("Answer:")
print(result["answer"])

In [12]:
# And the Pulp Fiction one. The Ezekiel speech is around the 15% mark of the
# script, just past our old 40K-char truncation window. Scene retrieval finds it.
result = manual_scene_rag(
    "In the Pulp Fiction script, what does Jules say about the path of the righteous man?"
)
print("Cypher generated:")
print(result["cypher"])
print()
print(f"Rows: {result['rows_shape']}")
print()
print("Answer:")
print(result["answer"])

Cypher generated:
MATCH (m:Movie {title: "Pulp Fiction"})-[:HAS_SCENE]->(sc:Scene)
WHERE sc.body CONTAINS "righteous man"
RETURN sc.heading, sc.body

Rows: (3, 2)

Answer:
Jules says about the path of the righteous man: "The path of the righteous man is beset on all sides by the inequities of the selfish and the tyranny of evil men. Blessed is he who, in the name of charity and good will, shepherds the weak through the valley of darkness, for he is truly his brother's keeper and the finder of lost children. And I will strike down upon thee with great vengeance and furious anger those who attempt to poison and destroy my brothers."


### What just changed

Four things, none of them about the LLM itself:

1. **The graph is doing real work now.** It's no longer just "find the right document." It's "find the right *passage*," using openCypher's `CONTAINS` for keyword filtering. That keyword filtering is genuinely cheap because Kuzu can do it on the indexed `Scene` table.

2. **Context shrank by 10x to 100x.** Each Get Out scene is ~3K characters; the whole script is 232K. Sending only the relevant scenes means we pay a fraction of the per-question cost *and* the LLM has a much higher signal-to-noise ratio. Less haystack, easier needle.

3. **Case-sensitivity papercut.** Kuzu's `CONTAINS` is case-sensitive. A query for `'sunken place'` won't match the script's `'Sunken Place'`. We worked around it by wrapping both sides with `lower()` and teaching the prompt to do the same. Without that, the first iteration of this section returned zero rows and the LLM honestly said "the text doesn't say."

4. **Long-phrase papercut.** Screenplays have unpredictable line breaks and indentation. `CONTAINS 'path of the righteous man'` returns zero rows because the script has the phrase wrapped across lines. `CONTAINS 'Ezekiel'` finds it instantly. Picking *one short distinctive keyword* per question is much more reliable than picking the full phrase. The prompt teaches this; in production you'd evolve into a vector search where the question itself becomes the query (module 05).

This is genuinely the production pattern for graph + text RAG. Chunk your text on whatever natural boundaries exist (scenes, sections, support tickets, paragraphs), attach the chunks to graph nodes, let the graph filter them, and let the LLM read the survivors.

### What scene-level retrieval doesn't do

It's keyword-based. If the question is "what is the sunken place?" and the script never uses those exact words (but does describe the concept), we miss it. The fix is **vector embeddings on the scene bodies**: instead of `CONTAINS 'sunken place'`, you embed the question and find the scenes whose embeddings are closest.

That's the hybrid pattern (graph + vector) and the main subject of [module 05](../05-advanced-rag/). NB4 has done the heavy lifting: chunked your documents, attached them to the graph, demonstrated chunk-level retrieval. Module 05 swaps the keyword filter for a vector search.

## Recap

The whole module compressed into one table:

| Notebook | What goes into the graph | Retrieval | What the LLM gets |
|---|---|---|---|
| NB1 | 3 nodes, 6 edges | (no LLM) | (no LLM) |
| NB2 | 30 nodes, 54 edges | (no LLM) | (no LLM) |
| NB3 | same as NB2 | rows of structured data | structured data, summarized |
| **NB4 Act II** | NB2 + 11 Script nodes (~2.4M chars) | one full script per query | the whole document, truncated |
| **NB4 Act III** | NB2 + 11 Script + 1,809 Scene nodes | the few scenes that match | only the relevant chunks |

Three things to take with you:

1. **The graph is the index, not the answer.** It picks the document, then the right chunk inside it. The chunks are still the chunks.
2. **Chunking is a schema decision.** We split on `INT./EXT.` because that's the natural unit in screenplays. Pick the natural unit in your domain (sections, support tickets, paragraphs, function bodies, whatever) and represent it as a node type.
3. **Keyword filtering on chunks is a fine baseline.** It scales much better than whole-document retrieval. The next step up is vector embeddings on the chunks, which lets you find conceptually-related scenes even when the keywords don't match. That's module 05.

## Where to go next

- **[Module 04: Evaluating RAG.](../04-evaluating-rag/)** Now that the answer comes from real prose, how do you tell if it's faithful to the source?
- **[Module 05: Advanced RAG.](../05-advanced-rag/)** The hybrid pattern: graph filters to candidate scenes, vector search ranks them by semantic similarity, the LLM answers from the top few. The full production answer.